# Week 8: Vector Workflows in Python

This notebook mirrors the vector analysis you did in **QGIS Week 3** (Vector Analysis & Attribute Joins).

**What you'll learn:**
- Load and clean spatial data
- Perform spatial joins (like Join Attributes by Location)
- Count points in polygons (like the Count Points in Polygon tool)
- Calculate density metrics (like Field Calculator)
- Create choropleth maps (like Graduated symbology)

---

## Python ↔ QGIS Week 3 Comparison

| QGIS Week 3 Operation | Python Equivalent |
|----------------------|-------------------|
| Load shapefile | `gpd.read_file("data.shp")` |
| Open Attribute Table | `gdf.head()` or `gdf` |
| Join Attributes by Location | `gpd.sjoin(points, polygons)` |
| Count Points in Polygon | `sjoin().groupby().size()` |
| Field Calculator (new field) | `gdf['new_col'] = expression` |
| Graduated Symbology | `.plot(column='field', scheme='quantiles')` |
| Export > Save Features As | `gdf.to_file("output.gpkg")` |

---

## Data options

| Option | Description |
|--------|-------------|
| **Sample data (default)** | NYC neighbourhoods + 311 complaints. No setup needed! |
| **Your own data** | Export from QGIS and place in `data/raw/` |

**Recommendation:** Start with sample data to learn the workflow, then try your own data.

---

## Step 0: Set up your environment

This cell detects your environment and installs packages.

**What's happening:**
- Check if we're in Google Colab or local Jupyter
- Install required packages if in Colab

In [ ]:
# ============================================================
# STEP 0: DETECT ENVIRONMENT AND INSTALL PACKAGES
# ============================================================
# Same pattern as Week 7 - this makes notebooks portable
# between Google Colab and local Jupyter environments.

import sys

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing GIS packages (takes ~1 minute)...")
    
    # Install required packages
    # geopandas: spatial data handling
    # contextily: add basemaps to plots
    # mapclassify: classification schemes for choropleth maps
    !pip install geopandas contextily mapclassify -q
    
    print("Done!")
else:
    print("Running in local Jupyter")
    print("Make sure you activated your conda environment: conda activate intro-gis")

---

## Step 1: Set up folder paths

**QGIS equivalent:** The folder structure you created in Week 1.

We use the same structure:
- `data/raw/` — Original input files (never modify)
- `data/processed/` — Your analysis outputs

In [ ]:
# ============================================================
# STEP 1: SET UP DATA PATHS
# ============================================================
# Same pattern as Week 7. This ensures your code works
# regardless of where you're running it.

from pathlib import Path

if IN_COLAB:
    # Mount Google Drive to access your files
    from google.colab import drive
    drive.mount('/content/drive')

    # Set paths to your data folders in Drive
    RAW = Path("/content/drive/MyDrive/intro-gis/week08/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week08/data/processed")
else:
    # Local paths (relative to notebook location)
    RAW = Path("data/raw")
    PROCESSED = Path("data/processed")

# Create folders if they don't exist
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder: {RAW}")
print(f"Processed folder: {PROCESSED}")

---

## Step 2: Import libraries

**Key libraries for vector analysis:**

| Library | Purpose | QGIS equivalent |
|---------|---------|------------------|
| `geopandas` | Spatial data operations | Vector menu, Processing tools |
| `pandas` | Data manipulation | Attribute table operations |
| `matplotlib` | Visualization | Print Layout |

In [ ]:
# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

import geopandas as gpd    # Spatial data (like shapefiles in memory)
import pandas as pd        # Tabular data (like CSV files)
import matplotlib.pyplot as plt  # Plotting and maps

print("Libraries imported successfully!")
print(f"GeoPandas version: {gpd.__version__}")

---

## Step 3: Load data

**QGIS equivalent:** Layer > Add Vector Layer

This cell checks for local data files. If none are found, it downloads sample data.

### Sample data (NYC)

The sample data uses **New York City**:
- **Neighbourhoods:** NYC Neighbourhood Tabulation Areas (NTAs) — polygon boundaries
- **Incidents:** 311 service requests — point locations of complaints/reports

This is the same type of analysis you did in **QGIS Week 3** with SA2 boundaries and SEIFA data!

### Using your own data

To use your own data, place these files in `data/raw/`:
- `neighbourhoods.geojson` — Polygon boundaries (e.g., SA2s from Week 3)
- `incidents.geojson` — Point locations (e.g., crime data from Week 5)

**To export from QGIS:**
1. Right-click layer > **Export** > **Save Features As...**
2. Format: **GeoJSON**
3. CRS: **EPSG:4326** (WGS 84)

In [ ]:
# ============================================================
# STEP 3: LOAD OR DOWNLOAD DATA
# ============================================================
# This pattern checks for local files first, then falls back
# to downloading sample data. This makes the notebook work
# "out of the box" while also supporting custom data.
#
# QGIS equivalent: Layer > Add Layer > Add Vector Layer

# Check if local files exist
local_neighbourhoods = RAW / "neighbourhoods.geojson"
local_incidents = RAW / "incidents.geojson"

if local_neighbourhoods.exists() and local_incidents.exists():
    # --------------------------------------------------------
    # USE LOCAL FILES (your own data from QGIS)
    # --------------------------------------------------------
    print("Found local data files - using your own data")
    print(f"  Neighbourhoods: {local_neighbourhoods}")
    print(f"  Incidents: {local_incidents}")
    
    neighbourhoods = gpd.read_file(local_neighbourhoods)
    incidents = gpd.read_file(local_incidents)
    USE_SAMPLE_DATA = False
    
else:
    # --------------------------------------------------------
    # DOWNLOAD NYC SAMPLE DATA
    # --------------------------------------------------------
    print("Local files not found - downloading NYC sample data...")
    print("(To use your own data, add neighbourhoods.geojson and")
    print(" incidents.geojson to data/raw/)\n")
    
    # NYC Neighbourhood Tabulation Areas (NTAs)
    # Source: NYC Open Data portal
    # These are like SA2 boundaries but for New York City
    nta_url = "https://data.cityofnewyork.us/api/geospatial/9nt8-h7nd?method=export&format=GeoJSON"
    print("Downloading NYC neighbourhoods (~2 seconds)...")
    neighbourhoods = gpd.read_file(nta_url)
    
    # NYC 311 Service Requests
    # These are complaints/reports from NYC residents (potholes,
    # noise, graffiti, etc.) - similar to crime/incident data
    # We limit to 5000 records for speed
    complaints_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.geojson?$limit=5000&$where=latitude IS NOT NULL"
    print("Downloading NYC 311 complaints (sample of 5,000)...")
    incidents = gpd.read_file(complaints_url)
    
    USE_SAMPLE_DATA = True
    print("\nDownload complete!")

# Report what we loaded
print(f"\nLoaded {len(neighbourhoods)} neighbourhoods (polygons)")
print(f"Loaded {len(incidents)} incidents (points)")
print(f"\nNeighbourhoods CRS: {neighbourhoods.crs}")
print(f"Incidents CRS: {incidents.crs}")

---

## Step 3b: Explore the data

**QGIS equivalent:** Opening the attribute table and using the Identify tool.

Always look at your data before analysis to understand its structure.

In [ ]:
# ============================================================
# STEP 3b: EXPLORE THE DATA
# ============================================================
# Before analysis, always inspect your data to understand:
# - What columns (fields) are available?
# - What are the data types?
# - Are there any obvious issues?
#
# QGIS equivalent: Right-click layer > Open Attribute Table

print("=" * 60)
print("NEIGHBOURHOODS (polygons)")
print("=" * 60)
print(f"Shape: {neighbourhoods.shape} (rows, columns)")
print(f"\nColumns: {list(neighbourhoods.columns)}")
print("\nFirst 3 rows:")
display(neighbourhoods.head(3))

In [ ]:
# Explore incidents data
print("=" * 60)
print("INCIDENTS (points)")
print("=" * 60)
print(f"Shape: {incidents.shape} (rows, columns)")
print(f"\nColumns: {list(incidents.columns)[:10]}...")  # First 10 columns
print("\nFirst 3 rows:")
display(incidents.head(3))

---

## Step 4: Clean and prepare data

**QGIS equivalent:** Field Calculator to standardize fields, Select by Expression to filter.

We need to:
1. Standardize column names (lowercase for consistency)
2. Identify the name column for joining
3. Calculate area in km² (need projected CRS first)

In [ ]:
# ============================================================
# STEP 4: CLEAN AND PREPARE DATA
# ============================================================
# Data cleaning is essential before analysis. Common tasks:
# - Standardize column names (avoid case-sensitivity issues)
# - Identify key columns for joining
# - Calculate derived fields
#
# QGIS equivalent: Field Calculator, Refactor Fields tool

# Convert column names to lowercase
# This prevents errors from inconsistent casing (e.g., 'Name' vs 'name')
neighbourhoods.columns = neighbourhoods.columns.str.lower()
incidents.columns = incidents.columns.str.lower()

# Identify the name column based on data source
# NYC data uses 'ntaname', Australian SA2 data uses 'sa2_name21'
if USE_SAMPLE_DATA:
    NAME_COL = 'ntaname'  # NYC neighbourhood names
else:
    # For Australian SA2 data - adjust if your column name differs
    NAME_COL = 'sa2_name21'
    if NAME_COL not in neighbourhoods.columns:
        print(f"Warning: '{NAME_COL}' not found.")
        print(f"Available columns: {list(neighbourhoods.columns)}")
        print("\nSet NAME_COL to your neighbourhood name column.")

print(f"Using '{NAME_COL}' as the neighbourhood name column")

# Calculate area in km²
# IMPORTANT: We must project to a CRS that uses meters first!
# EPSG:3857 is Web Mercator (meters), but distorts area at high latitudes
# For more accurate area, use a local projection (e.g., EPSG:7856 for Australia)
#
# QGIS equivalent: Field Calculator > $area / 1000000

# .to_crs(3857) temporarily reprojects for area calculation
# .area returns area in the CRS units (meters² for EPSG:3857)
# / 1e6 converts m² to km²
neighbourhoods["area_km2"] = neighbourhoods.to_crs(3857).area / 1e6

print(f"\nCalculated area for {len(neighbourhoods)} neighbourhoods")
print(f"Area range: {neighbourhoods['area_km2'].min():.2f} to {neighbourhoods['area_km2'].max():.2f} km²")

---

## Step 4b: Quick map to verify data

**QGIS equivalent:** Looking at the map canvas after adding layers.

Always visualize your data before analysis to catch any issues!

In [ ]:
# ============================================================
# STEP 4b: QUICK VISUALIZATION
# ============================================================
# Visual inspection catches issues that statistics miss:
# - Points outside boundaries?
# - Obvious data errors?
# - Unexpected patterns?
#
# QGIS equivalent: Looking at the map canvas

fig, ax = plt.subplots(figsize=(10, 10))

# Plot neighbourhoods (polygons) first - they form the base layer
# Like adding a polygon layer in QGIS with gray fill
neighbourhoods.plot(
    ax=ax, 
    color='lightgray',      # Fill color
    edgecolor='white',      # Border color
    linewidth=0.5           # Border thickness
)

# Plot incidents (points) on top
# Like adding a point layer in QGIS
incidents.plot(
    ax=ax, 
    color='red',            # Point color
    markersize=1,           # Point size (small because many points)
    alpha=0.3               # Transparency (0=invisible, 1=solid)
)

ax.set_title(f"Neighbourhoods ({len(neighbourhoods)}) and Incidents ({len(incidents)})")
ax.set_axis_off()  # Hide coordinate axes for cleaner look
plt.show()

print("\nCheck: Do the points fall within the neighbourhoods?")
print("If points are outside, the spatial join won't match them.")

---

## Step 5: Spatial join

**QGIS equivalent:** Vector > Data Management > Join Attributes by Location

A **spatial join** links features based on their geographic relationship (rather than a shared field value like an attribute join).

We want to count how many incidents fall within each neighbourhood.

**How it works:**
1. For each incident (point), find which neighbourhood (polygon) contains it
2. Link the incident to that neighbourhood's attributes
3. Group by neighbourhood and count incidents

**Key parameters:**
- `predicate="within"` — Point must be inside polygon
- `how="left"` — Keep all incidents, even if no match

In [ ]:
# ============================================================
# STEP 5: SPATIAL JOIN (Count Points in Polygons)
# ============================================================
# This is the core vector analysis operation!
# 
# QGIS equivalent: 
# - Vector > Data Management > Join Attributes by Location (Summary)
# - Vector > Analysis Tools > Count Points in Polygon
#
# The process:
# 1. gpd.sjoin() links each point to the polygon it falls within
# 2. .groupby().size() counts points per polygon
# 3. .merge() adds the counts back to the polygon layer

# Step 5a: Ensure both layers use the same CRS
# This is CRITICAL - spatial operations fail silently if CRS differs!
# QGIS equivalent: Checking layer CRS in Properties > Source
if neighbourhoods.crs != incidents.crs:
    print(f"CRS mismatch detected!")
    print(f"  Neighbourhoods: {neighbourhoods.crs}")
    print(f"  Incidents: {incidents.crs}")
    print("Reprojecting incidents to match neighbourhoods...")
    incidents = incidents.to_crs(neighbourhoods.crs)
    print(f"  Incidents now: {incidents.crs}")
else:
    print(f"CRS match confirmed: {neighbourhoods.crs}")

# Step 5b: Perform the spatial join
# gpd.sjoin() is like "Join Attributes by Location" in QGIS
#
# Parameters:
# - incidents: the left layer (points)
# - neighbourhoods: the right layer (polygons) 
# - predicate="within": point must be INSIDE polygon
# - how="left": keep all incidents even if no polygon match
print("\nPerforming spatial join...")
joined = gpd.sjoin(
    incidents,           # Left layer (points)
    neighbourhoods,      # Right layer (polygons)
    predicate="within",  # Spatial relationship
    how="left"           # Keep all left features
)

# Step 5c: Count incidents per neighbourhood
# .groupby(NAME_COL) groups rows by neighbourhood name
# .size() counts rows in each group
# .rename() gives the Series a name
#
# QGIS equivalent: Statistics by Categories, or Count Points in Polygon
counts = joined.groupby(NAME_COL).size().rename("incident_count")

print(f"Counted incidents in {len(counts)} neighbourhoods")

# Step 5d: Merge counts back to neighbourhoods
# .merge() is like a table join in QGIS
# We join on the NAME_COL field
neighbourhoods = neighbourhoods.merge(
    counts,           # The count data to add
    on=NAME_COL,      # The field to join on
    how="left"        # Keep all neighbourhoods even if count=0
)

# Step 5e: Fill missing values with 0
# Neighbourhoods with no incidents will have NaN (null)
# We replace these with 0
# QGIS equivalent: Field Calculator with coalesce() or CASE WHEN
neighbourhoods["incident_count"] = neighbourhoods["incident_count"].fillna(0)

print("\nSpatial join complete!")
print(f"\nTop 5 neighbourhoods by incident count:")
print(neighbourhoods.nlargest(5, "incident_count")[[NAME_COL, "incident_count"]].to_string(index=False))

---

## Step 6: Calculate incident rate

**QGIS equivalent:** Field Calculator to create a new field.

Raw counts are misleading because larger areas naturally have more incidents.
**Normalizing** by area gives a fair comparison.

**Rate = Count / Area**

This is the same concept as population density (people per km²).

In [ ]:
# ============================================================
# STEP 6: CALCULATE INCIDENT RATE
# ============================================================
# Normalizing counts by area allows fair comparison.
# A large rural area and a small urban area might have the
# same count, but very different rates!
#
# QGIS equivalent: Field Calculator
# Expression: "incident_count" / "area_km2"

# Calculate rate: incidents per square kilometer
neighbourhoods["rate_per_km2"] = (
    neighbourhoods["incident_count"] / neighbourhoods["area_km2"]
)

print("Calculated incident rate (per km²)")
print("\nTop 5 neighbourhoods by incident RATE:")
top_by_rate = neighbourhoods.nlargest(5, "rate_per_km2")[
    [NAME_COL, "incident_count", "area_km2", "rate_per_km2"]
]
print(top_by_rate.to_string(index=False))

print("\n" + "="*60)
print("Compare: Top 5 by COUNT vs Top 5 by RATE")
print("Notice how they're often different - that's why normalizing matters!")

---

## Step 7: Create a choropleth map

**QGIS equivalent:** Layer Properties > Symbology > Graduated

A **choropleth map** uses color to show values (like population density or incident rate).

**Key parameters:**
- `column="rate_per_km2"` — The field to visualize
- `scheme="quantiles"` — Classification method (like QGIS Mode)
- `k=5` — Number of classes
- `cmap="YlOrRd"` — Color ramp (Yellow-Orange-Red)

In [ ]:
# ============================================================
# STEP 7: CREATE CHOROPLETH MAP
# ============================================================
# Choropleth maps color polygons based on a data value.
# This is the Python equivalent of Graduated symbology in QGIS.
#
# QGIS equivalent:
# - Layer Properties > Symbology > Graduated
# - Value: rate_per_km2
# - Mode: Quantile (Equal Count)
# - Classes: 5
# - Color ramp: YlOrRd

fig, ax = plt.subplots(figsize=(12, 10))

# The .plot() method with column parameter creates a choropleth
neighbourhoods.plot(
    column="rate_per_km2",   # Field to visualize
    scheme="quantiles",      # Classification method
                             # Options: 'quantiles', 'equal_interval', 
                             # 'natural_breaks', 'fisher_jenks'
    k=5,                     # Number of classes
    cmap="YlOrRd",           # Color ramp (Yellow-Orange-Red)
                             # Other options: 'Blues', 'Greens', 'RdYlGn', 'viridis'
    legend=True,             # Add legend
    legend_kwds={            # Legend styling
        "title": "Incidents per km²",
        "loc": "lower right"
    },
    edgecolor="white",       # Polygon border color
    linewidth=0.3,           # Border thickness
    ax=ax                    # Draw on our axes
)

# Add title and remove axes
ax.set_title("Incident Rate by Neighbourhood", fontsize=14)
ax.set_axis_off()

plt.tight_layout()
plt.show()

print("\nThis map shows incident DENSITY (rate per km²).")
print("Darker colors = higher incident rates.")
print("\nIn QGIS, this would be Graduated symbology with Quantile classification.")

---

## Step 7b: Compare count vs rate

This side-by-side comparison shows why normalization matters.

In [ ]:
# ============================================================
# STEP 7b: COMPARE COUNT VS RATE MAPS
# ============================================================
# Side-by-side comparison shows how normalization changes
# the story the map tells.

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left map: Raw count (misleading - favors large areas)
neighbourhoods.plot(
    column="incident_count",
    scheme="quantiles",
    k=5,
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"title": "Count"},
    edgecolor="white",
    linewidth=0.3,
    ax=axes[0]
)
axes[0].set_title("Incident COUNT\n(raw numbers - can be misleading)")
axes[0].set_axis_off()

# Right map: Rate per km² (fair comparison)
neighbourhoods.plot(
    column="rate_per_km2",
    scheme="quantiles",
    k=5,
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"title": "Per km²"},
    edgecolor="white",
    linewidth=0.3,
    ax=axes[1]
)
axes[1].set_title("Incident RATE\n(per km² - fair comparison)")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

print("Notice how the patterns differ!")
print("Count maps favor large areas; rate maps show true density.")

---

## Step 8: Export for QGIS

**QGIS equivalent:** Right-click layer > Export > Save Features As

Save your results to `data/processed/` as a GeoPackage. You can then:
- Open in QGIS for professional cartography
- Share with colleagues
- Use in future notebooks

In [ ]:
# ============================================================
# STEP 8: EXPORT RESULTS
# ============================================================
# Save your analysis results so you can:
# - Open in QGIS for professional map design
# - Share with colleagues
# - Use in future analyses
#
# QGIS equivalent: Right-click layer > Export > Save Features As

# Create output path
output_path = PROCESSED / "neighbourhoods_with_incidents.gpkg"

# Save to GeoPackage (recommended format)
# GeoPackage is like a database - it can store multiple layers,
# preserves field types, and is widely supported
neighbourhoods.to_file(output_path, driver="GPKG")

print(f"Saved to: {output_path}")
print("\nYou can now open this file in QGIS!")
print("The file contains:")
print(f"  - {len(neighbourhoods)} neighbourhoods")
print(f"  - incident_count field (raw counts)")
print(f"  - rate_per_km2 field (normalized rate)")
print(f"  - area_km2 field (area in square kilometers)")

---

## Summary: Python ↔ QGIS Week 3

| What you did | Python | QGIS Week 3 |
|--------------|--------|-------------|
| Load data | `gpd.read_file()` | Add Vector Layer |
| Check CRS | `gdf.crs` | Layer Properties > Source |
| Reproject | `gdf.to_crs()` | Export > Save As with new CRS |
| Spatial join | `gpd.sjoin()` | Join Attributes by Location |
| Count by group | `.groupby().size()` | Count Points in Polygon |
| Calculate field | `gdf['new'] = expr` | Field Calculator |
| Choropleth map | `.plot(column=...)` | Graduated Symbology |
| Export | `gdf.to_file()` | Export > Save Features As |

### Key concepts

1. **Spatial join** = Linking features based on location (not attributes)
2. **Normalization** = Dividing by area for fair comparison
3. **Classification schemes** = How to group continuous data into classes
4. **Choropleth map** = Polygons colored by data values

---

## Try with your own data

1. **Export your Week 3 boundaries** from QGIS as `neighbourhoods.geojson`
2. **Export your Week 5 crime points** as `incidents.geojson`
3. Place both in `data/raw/`
4. Re-run this notebook!

The notebook will automatically detect and use your files.

---

## What's next?

**Week 9** covers raster analysis in Python:
- Calculate NDVI from satellite imagery
- Generate slope and aspect from DEMs
- Create hillshade visualizations

This mirrors what you did in **QGIS Week 4** (Raster & Terrain Analysis).

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`